### Import Dependencies

In [2]:
import openai
import pandas as pd
import tiktoken


from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import FieldCondition, MatchAny, Filter
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Create Qdrant collection for hybrid search

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

True

In [5]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [6]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [7]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data
- useful information can both from title and text, so concat it together

In [11]:
df_reviews = pd.read_json("../../data/raw/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [12]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Love it! It fits my Chromebook perfectly,Love it! It fits my Chromebook perfectly.,[],B00J39BYSG,B07BJ8KD6X,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2015-06-30 21:27:51.000,0,True
1,5,love this,i was concerned it wasnt gonna work with my TV...,[],B008GVOVK0,B008GVOVK0,AHGAOIZVODNHYMNCBV4DECZH42UQ,2018-08-24 22:21:36.983,0,True
2,5,nice product,adjustable and elastic,[],B01G54B68Y,B07HMRN7Y6,AGEP2CC2QIJU5IHQZXCC7BE5JSDQ,2019-11-12 17:33:07.317,0,True
3,4,Satisfied,Works well,[],B019WJROSK,B019WJROSK,AHSAUHUTLVNKJ72GUIPGI5XZH6ZA,2019-02-01 20:54:53.408,0,True
4,5,Works great,This wire works great. I recommend it. It's ...,[],B08G8PMN8C,B08J42GSW6,AHATA6X6MYTC3VNBFJ3WIYVK257A,2021-02-25 01:59:27.685,0,False


In [13]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [21]:
def count_tokens(row):

    encoding = tiktoken.encoding_for_model("text-embedding-3-small")

    return len(encoding.encode(row["preprocessed_data"]))

In [22]:
count_tokens({"preprocessed_data": "How are you doing today?"})

6

In [23]:
df_reviews["preprocessed_data"] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis=1)

In [28]:
len(df_reviews), len(df_reviews[df_reviews["token_count"] < 8192])

(137987, 137987)

In [ ]:
df_reviews = df_reviews[df_reviews["token_count"] < 8192] # only embedding model can only take 8192 tokens

In [29]:
total_tokens = df_reviews["token_count"].sum()
total_tokens

np.int64(8247850)

### Embed the text and add additional fields to the payload of each vector for reviews

In [33]:
df_data_to_embed = df_reviews[["preprocessed_data", "parent_asin"]].copy()
data_to_embed_reviews = df_data_to_embed.to_dict(orient="records")
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 137987
Processed 1000 of 137987
Processed 1500 of 137987
Processed 2000 of 137987
Processed 2500 of 137987
Processed 3000 of 137987
Processed 3500 of 137987
Processed 4000 of 137987
Processed 4500 of 137987
Processed 5000 of 137987
Processed 5500 of 137987
Processed 6000 of 137987
Processed 6500 of 137987
Processed 7000 of 137987
Processed 7500 of 137987
Processed 8000 of 137987
Processed 8500 of 137987
Processed 9000 of 137987
Processed 9500 of 137987
Processed 10000 of 137987
Processed 10500 of 137987
Processed 11000 of 137987
Processed 11500 of 137987
Processed 12000 of 137987
Processed 12500 of 137987
Processed 13000 of 137987
Processed 13500 of 137987
Processed 14000 of 137987
Processed 14500 of 137987
Processed 15000 of 137987
Processed 15500 of 137987
Processed 16000 of 137987
Processed 16500 of 137987
Processed 17000 of 137987
Processed 17500 of 137987
Processed 18000 of 137987
Processed 18500 of 137987
Processed 19000 of 137987
Processed 19500 of 137987
Proces

In [48]:
df_data_to_embed[df_data_to_embed.preprocessed_data.str.contains('bad')]

,preprocessed_data,parent_asin
37,Best Headphones Best sounding headphones I use...,B00JB31OUW
48,"Good Sounding, Poor Fitting (on me) I debated ...",B07PZQ8MKL
76,Other than that they are fine. nice picture Th...,B00V8FAWC2
94,Like this 180* design. I have only had it a da...,B088DC1KPP
160,the directions are pretty useless and I had to...,B000EHWCDW
...,...,...
137845,"Decent device, good sized ball, ok for noteboo...",B0038KLG5C
137846,"Decent device, good sized ball, ok for noteboo...",B0038KLG5C
137890,"BUY THE WARRANTY!!!!!! Good TV Picture, Though...",B0014175NE
137919,... help at all and screw hole placements are ...,B012GPCLR6


In [34]:
len(embeddings)

137987

In [35]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding # only use dense vector for embedding
            },
            payload=data
        )
    )
    i += 1

In [36]:
pointstructs[0].vector

{'text-embedding-3-small': [0.03839111328125,
  -0.040618896484375,
  -0.037200927734375,
  -0.028961181640625,
  -0.0009469985961914062,
  -0.07733154296875,
  -0.010284423828125,
  0.00091552734375,
  -0.046234130859375,
  0.0026798248291015625,
  -0.024322509765625,
  0.0256500244140625,
  -0.0278472900390625,
  0.007015228271484375,
  0.0472412109375,
  0.00269317626953125,
  -0.038177490234375,
  -0.0237884521484375,
  0.020416259765625,
  0.0257110595703125,
  -0.01316070556640625,
  0.03973388671875,
  0.015838623046875,
  -0.0316162109375,
  0.00476837158203125,
  -0.050811767578125,
  -0.02801513671875,
  0.0034656524658203125,
  0.0203704833984375,
  -0.054931640625,
  -0.0072174072265625,
  -0.02154541015625,
  -0.00518798828125,
  -0.05255126953125,
  -0.01490020751953125,
  -0.010528564453125,
  -0.00939178466796875,
  -0.026580810546875,
  -0.011566162109375,
  0.04901123046875,
  -0.0217742919921875,
  0.055511474609375,
  0.0880126953125,
  0.04998779296875,
  -0.080017

In [37]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 137987
Processed 200 of 137987
Processed 300 of 137987
Processed 400 of 137987
Processed 500 of 137987
Processed 600 of 137987
Processed 700 of 137987
Processed 800 of 137987
Processed 900 of 137987
Processed 1000 of 137987
Processed 1100 of 137987
Processed 1200 of 137987
Processed 1300 of 137987
Processed 1400 of 137987
Processed 1500 of 137987
Processed 1600 of 137987
Processed 1700 of 137987
Processed 1800 of 137987
Processed 1900 of 137987
Processed 2000 of 137987
Processed 2100 of 137987
Processed 2200 of 137987
Processed 2300 of 137987
Processed 2400 of 137987
Processed 2500 of 137987
Processed 2600 of 137987
Processed 2700 of 137987
Processed 2800 of 137987
Processed 2900 of 137987
Processed 3000 of 137987
Processed 3100 of 137987
Processed 3200 of 137987
Processed 3300 of 137987
Processed 3400 of 137987
Processed 3500 of 137987
Processed 3600 of 137987
Processed 3700 of 137987
Processed 3800 of 137987
Processed 3900 of 137987
Processed 4000 of 137987
Processed

### A function to run search against reviews on a prefiltered set of product IDs

In [51]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"), # no weighting
        limit=k
    )
    
    return results

In [52]:
reviews = retrieve_prefiltered_reviews_data("bad_quality", ["B012GPCLR6"])

In [53]:
reviews.points

[ScoredPoint(id=17392, version=176, score=0.5, payload={'preprocessed_data': '🤔 I must warn you The thin is very crapy quality', 'parent_asin': 'B012GPCLR6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=121456, version=1217, score=0.33333334, payload={'preprocessed_data': 'BAD DESIGN IN THIS REGARD ONE PROBLEM - YOU HAVE TO TAKE IT APART TO CHANGE MEMORY CARDS.<br />BAD DESIGN IN THIS REGARD....<br />NICE CASE EXCEPT FOR THIS ONE PROBLEM.....', 'parent_asin': 'B012GPCLR6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=56370, version=566, score=0.25, payload={'preprocessed_data': "oh kay not great Ordered 2.  1 of the cases the thread in screw eyelets was stripped on 3 of the 4 exterior holes. Also, one of the screw eyelets where you attach the board the the base wasn't drilled out.  The other of the 2 cases was fine.  2 stars for lack of quality assurance.", 'parent_asin': 'B012GPCLR6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint

In [54]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B012GPCLR6", "B0014175NE"])

In [55]:
reviews.points

[ScoredPoint(id=17392, version=176, score=0.5, payload={'preprocessed_data': '🤔 I must warn you The thin is very crapy quality', 'parent_asin': 'B012GPCLR6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=56370, version=566, score=0.33333334, payload={'preprocessed_data': "oh kay not great Ordered 2.  1 of the cases the thread in screw eyelets was stripped on 3 of the 4 exterior holes. Also, one of the screw eyelets where you attach the board the the base wasn't drilled out.  The other of the 2 cases was fine.  2 stars for lack of quality assurance.", 'parent_asin': 'B012GPCLR6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=89433, version=897, score=0.25, payload={'preprocessed_data': 'Poor quality When the item arrived I was dissapointed with the &#34;touch of red&#34; as it was barely noticable and once placed on the entertainment center it could not be seen at all.  After setting everything up, moving cords all around, and finally sitting to enj

### Define the reviews retrieval tool

In [56]:
def get_embedding(text, model="text-embedding-3-small"):
    """因为 OpenAI Embeddings API 支持一次输入多条文本，所以返回的 data 始终是列表。当前只传入一条文本："""
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }


def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, user review: {chunk}\n"

    return formatted_context


def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:

    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(
        query,
        parent_asins,
        top_k
    )
    formatted_context = process_reviews_context(retrieved_context)

    return formatted_context

In [58]:
result = get_formatted_reviews_context("bad quality", ["B012GPCLR6", "B0014175NE"])
print(result)

- ID: B012GPCLR6, user review: 🤔 I must warn you The thin is very crapy quality
- ID: B012GPCLR6, user review: oh kay not great Ordered 2.  1 of the cases the thread in screw eyelets was stripped on 3 of the 4 exterior holes. Also, one of the screw eyelets where you attach the board the the base wasn't drilled out.  The other of the 2 cases was fine.  2 stars for lack of quality assurance.
- ID: B0014175NE, user review: Poor quality When the item arrived I was dissapointed with the &#34;touch of red&#34; as it was barely noticable and once placed on the entertainment center it could not be seen at all.  After setting everything up, moving cords all around, and finally sitting to enjoy the new flatscreen I noticed some thick black lines.  I got up to inspect and found that it was permanent marker drawn on the screen of the television.  Yes there are ways to remove this but already being dissapointed with the color I decided to return.  The reason I gave Samsung 2 stars is for making the